# HealthTwin: train the remaining models on Google Colab
Trains **sepsis (ML), sepsis causal Transformer + LSTM, forecasting and ECG** on the real public PhysioNet data (Colab can reach PhysioNet; many sandboxes cannot).
Use **Runtime > Change runtime type > T4 GPU** (optional, speeds up the Transformer). Takes roughly 30-90 minutes.
Nothing here is simulated: if a download fails the cell stops with an error.

## 1. Upload `healthtwin_FINAL.zip`

In [ ]:
from google.colab import files
up = files.upload()            # choose healthtwin_FINAL.zip
import zipfile, os
zipfile.ZipFile(next(iter(up))).extractall('.')
%cd healthtwin
!pip -q install -r requirements.txt

## 2. Sepsis data (PhysioNet/CinC 2019) and the ML model

In [ ]:
!python scripts/download_sepsis.py --limit 3000 --workers 8
!python scripts/prepare_sepsis_data.py
!python scripts/train_sepsis.py

## 3. Causal Transformer + LSTM (same patient-wise split)

In [ ]:
!python scripts/train_causal_transformer.py --epochs 8 --batch-size 256

## 4. Vital-sign forecasting and ECG (MIT-BIH)

In [ ]:
!python scripts/train_forecast.py
!python scripts/download_ecg.py
!python scripts/train_ecg.py

## 5. Check the gates, then download the results

In [ ]:
!python scripts/check_environment.py 2>&1 | grep -E "model |origin"
!zip -qr trained_results.zip artifacts reports data/processed
files.download('trained_results.zip')

## 6. Back on your laptop
Unzip `trained_results.zip` **into the project folder** (merge `artifacts/`, `reports/` and `data/processed/`), then run `python scripts/check_environment.py` and `streamlit run app.py`.
A model shows as available only when it has real-data provenance and a matching evaluation report - copying files without both keeps it unavailable.